# 05 - Experimentos K en RITA (No Supervisado)

Notebook para comparar `K=4,5,6,7,8` con K-Modes, ver geometria de clusters (PCA/t-SNE), radar de perfiles, composicion por cluster y transicion de clusters heredados.

In [4]:
%pip install kmodes scikit-learn pandas matplotlib seaborn numpy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kmodes.kmodes import KModes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'reports').exists():
            return p
    raise RuntimeError('No se encontro la raiz del proyecto')

ROOT = find_project_root(Path.cwd().resolve())
print('Project root:', ROOT)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Project root: /home/raul/Documents/Universidad/introduccion-a-machine-learning-course-unsa


In [ ]:
# Rutas
input_path = ROOT / 'data/processed/rita_limpio.csv'
out_dir = ROOT / 'reports/k_experimentos'
fig_dir = out_dir / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

# Verificar si el archivo existe
if not input_path.exists():
	print(f'Error: El archivo no existe en {input_path}')
	print('Rutas disponibles en data/processed/:')
	data_dir = ROOT / 'data/processed'
	if data_dir.exists():
		print([f.name for f in data_dir.glob('*.csv')])
	else:
		print(f'La carpeta {data_dir} no existe.')
else:
	# Carga
	raw = pd.read_csv(input_path)
	required = ['PTESXN', 'TOPOGRAFÍA_N', 'MORFOLOGÍA_N', 'EDAD_DIAGNÓSTICO']
	base = raw[required].dropna().copy()

	# Variable discretizada de edad (como en analisis previo)
	bins = [-1, 18, 45, 65, 200]
	labels = ['Infantil-Juvenil', 'Adulto Joven', 'Adulto', 'Adulto Mayor']
	base['EDAD_RANGO'] = pd.cut(base['EDAD_DIAGNÓSTICO'], bins=bins, labels=labels).astype(str)

	cluster_features = ['PTESXN', 'TOPOGRAFÍA_N', 'MORFOLOGÍA_N', 'EDAD_RANGO']
	X_cat = base[cluster_features].astype(str).copy()

	print('Filas usadas:', len(base))
	X_cat.head(2)

Error: El archivo no existe en /home/raul/Documents/Universidad/introduccion-a-machine-learning-course-unsa/data/processed/rita_limpio.csv
Rutas disponibles en data/processed/:
La carpeta /home/raul/Documents/Universidad/introduccion-a-machine-learning-course-unsa/data/processed no existe.


NameError: name 'raw' is not defined

In [ ]:
# Codificacion para K-Modes
X_enc = X_cat.copy()
enc_maps = {}
for c in X_enc.columns:
    vals = sorted(X_enc[c].unique().tolist())
    mp = {v: i for i, v in enumerate(vals)}
    enc_maps[c] = mp
    X_enc[c] = X_enc[c].map(mp)

# Muestra para geometria (PCA/t-SNE) para acelerar visualizacion
n_total = len(X_cat)
sample_n = min(12000, n_total)
rng = np.random.default_rng(42)
sample_pos = np.sort(rng.choice(np.arange(n_total), size=sample_n, replace=False))

X_cat_sample = X_cat.iloc[sample_pos].copy()
X_geom_sample = pd.get_dummies(X_cat_sample, columns=cluster_features, drop_first=False)
X_scaled_sample = StandardScaler().fit_transform(X_geom_sample)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_sample)

print(f'Computando t-SNE en muestra de {sample_n} puntos...')
tsne = TSNE(n_components=2, perplexity=35, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_scaled_sample)

print('Varianza explicada PCA 2D:', round(float(pca.explained_variance_ratio_.sum()*100), 2), '%')

In [ ]:
# Barrido de K y guardado de resultados
k_values = [4, 5, 6, 7, 8]
colors = ['#3a86ff', '#ff006e', '#ffbe0b', '#8338ec', '#06d6a0', '#fb5607', '#2a9d8f', '#9b5de5']

summary = {
    'dataset_rows_used': int(n_total),
    'sample_rows_for_geometry': int(sample_n),
    'features': cluster_features,
    'k_results': {}
}

cluster_assignments = pd.DataFrame(index=base.index)

for k in k_values:
    print(f'Entrenando K-Modes con K={k}...')
    km = KModes(n_clusters=k, init='Huang', n_init=8, random_state=42)
    clusters_full = km.fit_predict(X_enc)

    work = base.copy()
    work['cluster'] = clusters_full
    cluster_assignments[f'K{k}'] = clusters_full

    sizes = work['cluster'].value_counts().sort_index().to_dict()
    centroides = []
    for i, row in enumerate(km.cluster_centroids_.astype(int)):
        item = {'cluster': int(i)}
        for j, c in enumerate(X_enc.columns):
            inv = {v: kk for kk, v in enc_maps[c].items()}
            item[c] = inv.get(int(row[j]), str(row[j]))
        centroides.append(item)

    summary['k_results'][str(k)] = {
        'cost': float(km.cost_),
        'sizes': {str(a): int(b) for a, b in sizes.items()},
        'centroids': centroides
    }

    # CSV por K
    work.to_csv(out_dir / f'rita_clusters_k{k}.csv', index=False)

summary

In [ ]:
# Visualizaciones por cada K
for k in k_values:
    print(f'Generando figuras K={k}...')
    work = base.copy()
    work['cluster'] = cluster_assignments[f'K{k}'].values

    # Sample labels for geometry
    clusters_s = cluster_assignments[f'K{k}'].values[sample_pos]
    cluster_ids = sorted(np.unique(clusters_s))

    # 1) Geometria PCA + t-SNE
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'RITA - Geometria de Clusters (K={k}, muestra={sample_n})', fontsize=14, fontweight='bold')

    for idx, cid in enumerate(cluster_ids):
        color = colors[idx % len(colors)]
        mask = clusters_s == cid
        axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, alpha=0.4, s=9, label=f'Cluster {cid}')

    centroids_pca = np.array([X_pca[clusters_s == cid].mean(axis=0) for cid in cluster_ids])
    axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='white', s=140, marker='X', zorder=5,
                    edgecolors='black', linewidth=1, label='Centroides')
    axes[0].set_title(f'PCA 2D (varianza: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].legend(framealpha=0.3, fontsize=8)

    for idx, cid in enumerate(cluster_ids):
        color = colors[idx % len(colors)]
        mask = clusters_s == cid
        axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color, alpha=0.4, s=9, label=f'Cluster {cid}')

    axes[1].set_title('t-SNE 2D (estructura local)')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    axes[1].legend(framealpha=0.3, fontsize=8)

    plt.tight_layout()
    plt.savefig(fig_dir / f'rita_cluster_geometry_k{k}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 2) Radar de perfiles
    prof = work.copy()
    prof['sexo_num'] = prof['PTESXN'].map({'Hombre': 0, 'Mujer': 1}).fillna(0)
    prof['topografia_code'] = prof['TOPOGRAFÍA_N'].astype('category').cat.codes
    prof['morfologia_code'] = prof['MORFOLOGÍA_N'].astype('category').cat.codes
    prof['edad_rango_code'] = prof['EDAD_RANGO'].map({
        'Infantil-Juvenil': 0, 'Adulto Joven': 1, 'Adulto': 2, 'Adulto Mayor': 3
    }).fillna(0)

    radar_features = ['EDAD_DIAGNÓSTICO', 'sexo_num', 'topografia_code', 'morfologia_code', 'edad_rango_code']
    radar_labels = ['Edad', 'Sexo', 'Topografia', 'Morfologia', 'Rango Edad']

    cm = prof.groupby('cluster')[radar_features].mean()
    den = (cm.max() - cm.min()).replace(0, 1)
    cmn = (cm - cm.min()) / den

    N = len(radar_features)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    rows = int(np.ceil(k / 2))
    cols = 2
    fig, axes = plt.subplots(rows, cols, figsize=(13, 4.8 * rows), subplot_kw=dict(polar=True))
    fig.suptitle(f'RITA - Radar de Perfiles por Cluster (K={k})', fontsize=14, fontweight='bold')

    axes_arr = np.array(axes).reshape(-1)
    for idx, cid in enumerate(sorted(work['cluster'].unique())):
        ax = axes_arr[idx]
        values = cmn.loc[cid].values.tolist()
        values += values[:1]
        color = colors[idx % len(colors)]
        ax.plot(angles, values, 'o-', linewidth=2, color=color)
        ax.fill(angles, values, alpha=0.25, color=color)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(radar_labels, size=9)
        ax.set_ylim(0, 1)
        ax.set_title(f'Cluster {cid}', size=11, color=color, pad=12)
        ax.grid(linestyle='--', alpha=0.6)

    for j in range(len(sorted(work['cluster'].unique())), len(axes_arr)):
        axes_arr[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(fig_dir / f'rita_radar_profiles_k{k}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # 3) Composicion por cluster (sexo y rango edad)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    fig.suptitle(f'RITA - Composicion por Cluster (K={k})', fontsize=14, fontweight='bold')

    sexo_cross = work.groupby(['cluster', 'PTESXN']).size().unstack(fill_value=0)
    sexo_pct = sexo_cross.div(sexo_cross.sum(axis=1), axis=0) * 100
    sexo_pct.plot(kind='bar', ax=axes[0], edgecolor='black', width=0.65)
    axes[0].set_title('Distribucion de Sexo (%)')
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('Porcentaje')
    axes[0].tick_params(axis='x', rotation=0)

    edad_cross = work.groupby(['cluster', 'EDAD_RANGO']).size().unstack(fill_value=0)
    edad_pct = edad_cross.div(edad_cross.sum(axis=1), axis=0) * 100
    edad_pct.plot(kind='bar', ax=axes[1], edgecolor='black', width=0.65)
    axes[1].set_title('Distribucion de Rango de Edad (%)')
    axes[1].set_xlabel('Cluster')
    axes[1].set_ylabel('Porcentaje')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig(fig_dir / f'rita_demographic_breakdown_k{k}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Analisis de clusters heredados (transicion entre K consecutivos)
pairs = [(4,5), (5,6), (6,7), (7,8)]

for k1, k2 in pairs:
    c1 = f'K{k1}'
    c2 = f'K{k2}'
    trans = pd.crosstab(cluster_assignments[c1], cluster_assignments[c2], normalize='index') * 100

    plt.figure(figsize=(8, 5.2))
    sns.heatmap(trans, annot=True, fmt='.1f', cmap='Blues')
    plt.title(f'Transicion de Clusteres Heredados: K={k1} -> K={k2} (%)')
    plt.xlabel(f'Cluster en K={k2}')
    plt.ylabel(f'Cluster en K={k1}')
    plt.tight_layout()
    plt.savefig(fig_dir / f'rita_transicion_k{k1}_k{k2}.png', dpi=150)
    plt.show()

    # mapping dominante de herencia
    mapping = trans.idxmax(axis=1)
    share = trans.max(axis=1).round(2)
    heredado = pd.DataFrame({'cluster_origen': mapping.index, 'cluster_destino_dominante': mapping.values, 'porcentaje_dominante': share.values})
    print(f'Herencia dominante K={k1} -> K={k2}')
    display(heredado)

In [ ]:
# Guardado de resumen
tabla = []
for k in k_values:
    r = summary['k_results'][str(k)]
    tabla.append({'K': k, 'Costo': r['cost'], 'Num_Clusters': len(r['sizes']), 'Tamano_Min': min(r['sizes'].values()), 'Tamano_Max': max(r['sizes'].values())})

resumen_df = pd.DataFrame(tabla)
display(resumen_df)

(out_dir / 'resumen_k_experimentos.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
resumen_df.to_csv(out_dir / 'resumen_tabla_k_experimentos.csv', index=False)

md_lines = []
md_lines.append('# Experimentos K-Modes en RITA (K=4..8)')
md_lines.append('')
md_lines.append('Resumen automatico generado desde notebook de experimentacion.')
md_lines.append('')
for _, row in resumen_df.iterrows():
    md_lines.append(f"- K={int(row['K'])}: costo={row['Costo']:.1f}, tamano_min={int(row['Tamano_Min'])}, tamano_max={int(row['Tamano_Max'])}")

(out_dir / 'REPORTE_K_EXPERIMENTOS.md').write_text('
'.join(md_lines), encoding='utf-8')
print('Archivos de resumen actualizados en:', out_dir)